# SimFin Quality & Review

Value-level data quality checks: accounting identities, cross-statement consistency, plausibility.

Tolerance: relative diff > 0.5% (configurable in `irp.quality.rules.violations`).

After the overview, step through unreviewed findings one by one in the Review section. Reviews persist in `data/data_quality/anomaly_reviews.toml`.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pandas.io.formats.style_render import ExtFormatter

from irp.data.simfin import statement
from irp.quality.simfin_rules import REGISTRY
from irp.quality.simfin_runner import run
from irp.quality.simfin_inspect import inspect
from irp.quality.simfin_reviews import add_review, add_flag, load_reviews_df, load_flags_df

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

SAMPLE = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'JPM', 'XOM', 'JNJ', 'WMT']
VARIANT = 'A'


def _edgar_link(url):
    return f'<a href="{url}" target="_blank">view</a>' if url else ''


def _edgar_html(url) -> str:
    if pd.notna(url) and url:
        return f'<a href="{url}" target="_blank">open EDGAR</a>'
    return '<span style="color:#aaa">(no EDGAR url)</span>'


def _fmt_num(x):
    if pd.isna(x):
        return ''
    try:
        return f'{float(x):,.0f}'
    except (TypeError, ValueError):
        return str(x)


_NUM_FMT: ExtFormatter = {
    'LHS_value': '{:,.0f}',
    'RHS_value': '{:,.0f}',
    'diff': '{:,.0f}',
    'rel_diff': '{:.2%}',
    'EDGAR': _edgar_link,
}

## Registered Rules

In [2]:
_rules_df = pd.DataFrame([
    {'Rule': r.name, 'Statement': r.statement, 'LHS': r.lhs, 'RHS': r.rhs}
    for r in REGISTRY
])
display(_rules_df)

,Rule,Statement,LHS,RHS
0,assets_split,balance,Total Assets,Total Current Assets + Total Noncurrent Assets
1,liab_split,balance,Total Liabilities,Total Current Liabilities + Total Noncurrent L...
2,liab_equity_split,balance,Total Liabilities & Equity,Total Liabilities + Total Equity
3,accounting_equation,balance,Total Assets,Total Liabilities & Equity
4,gross_profit,income,Gross Profit,Revenue + Cost of Revenue
5,operating_income,income,Operating Income (Loss),Gross Profit + Operating Expenses
6,continuing_ops,income,Income (Loss) from Continuing Operations,Pretax Income (Loss) + Income Tax (Expense) Be...
7,net_income_split,income,Net Income,Income (Loss) from Continuing Operations + Net...
8,cash_chain,cashflow,Net Change in Cash,CFO + CFI + CFF
9,ni_income_vs_cashflow,cross,Net Income (income),Net Income/Starting Line (cashflow)


## Annual

In [3]:
findings_A = run(SAMPLE, 'A')
print(f'Total findings: {len(findings_A)}')
display(findings_A.drop(columns=['CIK'], errors='ignore').style.format(_NUM_FMT))

Total findings: 28


,Rule,Statement,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Period_str,Report Date,EDGAR
0,cash_chain,cashflow,META,2024,FY,A,"2,611,000,000","3,397,000,000","-786,000,000",23.14%,2024FY,2024-12-31 00:00:00,view
1,cash_chain,cashflow,META,2022,FY,A,"-1,269,000,000","-631,000,000","-638,000,000",50.28%,2022FY,2022-12-31 00:00:00,view
2,cash_chain,cashflow,META,2021,FY,A,"-1,089,000,000","-615,000,000","-474,000,000",43.53%,2021FY,2021-12-31 00:00:00,view
3,cash_chain,cashflow,META,2020,FY,A,"-1,325,000,000","-1,604,000,000","279,000,000",17.39%,2020FY,2020-12-31 00:00:00,view
4,cash_chain,cashflow,MSFT,2024,FY,A,"-16,389,000,000","-16,179,000,000","-210,000,000",1.28%,2024FY,2024-06-30 00:00:00,view
5,cash_chain,cashflow,MSFT,2023,FY,A,"20,773,000,000","20,967,000,000","-194,000,000",0.93%,2023FY,2023-06-30 00:00:00,view
6,cash_chain,cashflow,MSFT,2022,FY,A,"-293,000,000","-152,000,000","-141,000,000",48.12%,2022FY,2022-06-30 00:00:00,view
7,cash_chain,cashflow,MSFT,2021,FY,A,"648,000,000","677,000,000","-29,000,000",4.28%,2021FY,2021-06-30 00:00:00,view
8,cash_chain,cashflow,MSFT,2020,FY,A,"2,220,000,000","2,421,000,000","-201,000,000",8.30%,2020FY,2020-06-30 00:00:00,view
9,cash_chain,cashflow,WMT,2024,FY,A,"-399,000,000","242,000,000","-641,000,000",160.65%,2024FY,2025-01-31 00:00:00,view


## Quarterly

In [4]:
findings_Q = run(SAMPLE, 'Q')
print(f'Total findings: {len(findings_Q)}')
display(findings_Q.drop(columns=['CIK'], errors='ignore').style.format(_NUM_FMT))

Total findings: 147


,Rule,Statement,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Period_str,Report Date,EDGAR
0,cash_chain,cashflow,AMZN,2025,Q1,Q,"-12,419,000,000","-12,835,000,000","416,000,000",3.24%,2025Q1,2025-03-31 00:00:00,view
1,cash_chain,cashflow,AMZN,2024,Q1,Q,"-558,000,000","-129,000,000","-429,000,000",76.88%,2024Q1,2024-03-31 00:00:00,view
2,cash_chain,cashflow,AMZN,2024,Q2,Q,"-1,659,000,000","-1,347,000,000","-312,000,000",18.81%,2024Q2,2024-06-30 00:00:00,view
3,cash_chain,cashflow,AMZN,2024,Q3,Q,"7,004,000,000","6,314,000,000","690,000,000",9.85%,2024Q3,2024-09-30 00:00:00,view
4,cash_chain,cashflow,AMZN,2024,Q4,Q,"3,635,000,000","4,885,000,000","-1,250,000,000",25.59%,2024Q4,2024-12-31 00:00:00,view
5,cash_chain,cashflow,AMZN,2023,Q1,Q,"-4,519,000,000","-4,664,000,000","145,000,000",3.11%,2023Q1,2023-03-31 00:00:00,view
6,cash_chain,cashflow,AMZN,2023,Q2,Q,"333,000,000","264,000,000","69,000,000",20.72%,2023Q2,2023-06-30 00:00:00,view
7,cash_chain,cashflow,AMZN,2023,Q3,Q,"14,000,000","516,000,000","-502,000,000",97.29%,2023Q3,2023-09-30 00:00:00,view
8,cash_chain,cashflow,AMZN,2023,Q4,Q,"23,809,000,000","23,118,000,000","691,000,000",2.90%,2023Q4,2023-12-31 00:00:00,view
9,cash_chain,cashflow,AMZN,2022,Q1,Q,"122,000,000","106,000,000","16,000,000",13.11%,2022Q1,2022-03-31 00:00:00,view


## Findings by Rule

In [5]:
if len(findings_A):
    display(findings_A.groupby(['Rule', 'Ticker']).size().unstack(fill_value=0))
else:
    print('No annual findings.')

Ticker,META,MSFT,WMT,XOM
Rule,,,,
cash_chain,4,5,5,4
ni_income_vs_cashflow,0,0,5,5


## Findings by Ticker

In [6]:
if len(findings_A):
    display(findings_A.groupby('Ticker').size().sort_values(ascending=False))
else:
    print('No annual findings.')

Ticker
WMT     10
XOM      9
MSFT     5
META     4
dtype: int64

## Detail per Rule

In [7]:
for _rule_name, _group in findings_A.groupby('Rule'):
    print(f'=== {_rule_name} ({len(_group)} violations) ===')
    display(_group.drop(columns=['Rule', 'Statement', 'CIK'], errors='ignore').style.format(_NUM_FMT))
    print()

=== cash_chain (18 violations) ===


,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Period_str,Report Date,EDGAR
0,META,2024,FY,A,"2,611,000,000","3,397,000,000","-786,000,000",23.14%,2024FY,2024-12-31 00:00:00,view
1,META,2022,FY,A,"-1,269,000,000","-631,000,000","-638,000,000",50.28%,2022FY,2022-12-31 00:00:00,view
2,META,2021,FY,A,"-1,089,000,000","-615,000,000","-474,000,000",43.53%,2021FY,2021-12-31 00:00:00,view
3,META,2020,FY,A,"-1,325,000,000","-1,604,000,000","279,000,000",17.39%,2020FY,2020-12-31 00:00:00,view
4,MSFT,2024,FY,A,"-16,389,000,000","-16,179,000,000","-210,000,000",1.28%,2024FY,2024-06-30 00:00:00,view
5,MSFT,2023,FY,A,"20,773,000,000","20,967,000,000","-194,000,000",0.93%,2023FY,2023-06-30 00:00:00,view
6,MSFT,2022,FY,A,"-293,000,000","-152,000,000","-141,000,000",48.12%,2022FY,2022-06-30 00:00:00,view
7,MSFT,2021,FY,A,"648,000,000","677,000,000","-29,000,000",4.28%,2021FY,2021-06-30 00:00:00,view
8,MSFT,2020,FY,A,"2,220,000,000","2,421,000,000","-201,000,000",8.30%,2020FY,2020-06-30 00:00:00,view
9,WMT,2024,FY,A,"-399,000,000","242,000,000","-641,000,000",160.65%,2024FY,2025-01-31 00:00:00,view



=== ni_income_vs_cashflow (10 violations) ===


,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Period_str,Report Date,EDGAR
10,WMT,2024,FY,A,"19,436,000,000","20,157,000,000","-721,000,000",3.58%,2024FY,2025-01-31 00:00:00,view
12,WMT,2023,FY,A,"15,511,000,000","16,270,000,000","-759,000,000",4.67%,2023FY,2024-01-31 00:00:00,view
14,WMT,2022,FY,A,"11,680,000,000","11,292,000,000","388,000,000",3.32%,2022FY,2023-01-31 00:00:00,view
16,WMT,2021,FY,A,"13,673,000,000","13,940,000,000","-267,000,000",1.92%,2021FY,2022-01-31 00:00:00,view
18,WMT,2020,FY,A,"13,510,000,000","13,706,000,000","-196,000,000",1.43%,2020FY,2021-01-31 00:00:00,view
20,XOM,2024,FY,A,"33,680,000,000","35,063,000,000","-1,383,000,000",3.94%,2024FY,2024-12-31 00:00:00,view
22,XOM,2023,FY,A,"36,010,000,000","37,354,000,000","-1,344,000,000",3.60%,2023FY,2023-12-31 00:00:00,view
23,XOM,2022,FY,A,"55,740,000,000","57,577,000,000","-1,837,000,000",3.19%,2022FY,2022-12-31 00:00:00,view
25,XOM,2021,FY,A,"23,040,000,000","23,598,000,000","-558,000,000",2.36%,2021FY,2021-12-31 00:00:00,view
27,XOM,2020,FY,A,"-22,440,000,000","-23,251,000,000","811,000,000",3.49%,2020FY,2020-12-31 00:00:00,view


---

## Review Queue

In [8]:
all_findings = run(SAMPLE, VARIANT, skip_reviewed=True)
done: set[tuple[str, str, str]] = set()
state = {'idx': 0, 'sort': 'rel_diff_desc', 'min_rel': 0.0, 'rules': None}

sort_w = widgets.Dropdown(
    options=['rel_diff_desc', 'ticker', 'rule'],
    value='rel_diff_desc',
    description='Sort:',
)
min_rel_w = widgets.FloatText(value=0.0, description='Min rel:', layout=widgets.Layout(width='160px'))
_rule_options = sorted(all_findings['Rule'].unique()) if len(all_findings) else []
rules_w = widgets.SelectMultiple(
    options=_rule_options,
    description='Rules:',
    rows=min(6, max(1, len(_rule_options))),
    layout=widgets.Layout(width='400px'),
)
apply_btn = widgets.Button(description='Apply', button_style='info')

print(f'{len(all_findings)} unreviewed findings loaded')

28 unreviewed findings loaded


In [9]:
def _key(row):
    return (row['Ticker'], row['Period_str'], row['Rule'])


def _rebuild_queue():
    df = all_findings
    if done:
        keep = [_key(r) not in done for _, r in df.iterrows()]
        df = df[keep]
    if state['rules']:
        df = df[df['Rule'].isin(state['rules'])]
    if state['min_rel'] > 0:
        df = df[df['rel_diff'] >= state['min_rel']]
    if state['sort'] == 'rel_diff_desc':
        df = df.sort_values('rel_diff', ascending=False)
    elif state['sort'] == 'ticker':
        df = df.sort_values(['Ticker', 'Fiscal Year', 'Rule'], ascending=[True, False, True])
    elif state['sort'] == 'rule':
        df = df.sort_values(['Rule', 'Ticker', 'Fiscal Year'], ascending=[True, True, False])
    return df.reset_index(drop=True)


def _siblings(current_row, queue):
    t, ps, rule = _key(current_row)
    same_rule = queue[
        (queue['Ticker'] == t) & (queue['Rule'] == rule) & (queue['Period_str'] != ps)
    ]
    same_filing = queue[
        (queue['Ticker'] == t) & (queue['Period_str'] == ps) & (queue['Rule'] != rule)
    ]
    return same_rule, same_filing


def _make_cb_row(row, ticked):
    label = f"{row['Ticker']} {row['Period_str']}  {row['Rule']}  ({row['rel_diff']:.2%})"
    cb = widgets.Checkbox(value=ticked, description=label, indent=False,
                          layout=widgets.Layout(width='500px'))
    link = widgets.HTML(value=_edgar_html(row.get('EDGAR')))
    return widgets.HBox([cb, link]), cb


def _render_queue_list(max_rows: int = 50):
    if len(queue) == 0:
        queue_list_html.value = '<i>No unreviewed findings.</i>'
        return
    show = queue[['Ticker', 'Period_str', 'Rule', 'rel_diff']].head(max_rows).copy()
    show.insert(0, '', ['→' if i == state['idx'] else '' for i in range(len(show))])
    show['rel_diff'] = show['rel_diff'].map(lambda x: f'{x:.2%}')
    header = f'<p><b>{len(queue)} unreviewed</b>'
    if len(queue) > max_rows:
        header += f' (showing top {max_rows})'
    header += '</p>'
    queue_list_html.value = header + show.to_html(index=False, escape=False)


def render():
    global queue
    if state['idx'] >= len(queue):
        container.children = (widgets.HTML('<i>Queue empty for current filter.</i>'),)
        sibling_state['checkboxes'] = []
        _render_queue_list()
        return

    row = queue.iloc[state['idx']]
    progress = f'{state["idx"] + 1}/{len(queue)}'

    children = [
        widgets.HTML(
            f'<h3>[{progress}] {row["Ticker"]} {row["Period_str"]} — {row["Rule"]}</h3>'
            f'<p><b>{row["LHS_value"]:,.0f}</b> vs <b>{row["RHS_value"]:,.0f}</b> '
            f'(diff {row["diff"]:,.0f}, {row["rel_diff"]:.2%}) &nbsp;|&nbsp; {_edgar_html(row.get("EDGAR"))}</p>'
        )
    ]

    ins = inspect(
        row['Rule'], row['Ticker'],
        int(row['Fiscal Year']), str(row['Fiscal Period']), str(row['Period']),
    )
    if len(ins):
        num_cols = [c for c in ins.columns if ins[c].dtype.kind in 'fi']
        children.append(widgets.HTML(ins.style.format({c: _fmt_num for c in num_cols}).to_html()))

    same_rule, same_filing = _siblings(row, queue)
    cbs = []
    if len(same_rule):
        children.append(widgets.HTML(f'<b>Same rule, other periods ({len(same_rule)}) — pre-ticked:</b>'))
        for _, sib in same_rule.iterrows():
            hbox, cb = _make_cb_row(sib, ticked=True)
            cbs.append((cb, sib))
            children.append(hbox)
    if len(same_filing):
        children.append(widgets.HTML(f'<b>Same filing, other rules ({len(same_filing)}) — unticked:</b>'))
        for _, sib in same_filing.iterrows():
            hbox, cb = _make_cb_row(sib, ticked=False)
            cbs.append((cb, sib))
            children.append(hbox)

    container.children = tuple(children)
    sibling_state['checkboxes'] = cbs
    note_w.value = ''
    _render_queue_list()


def on_next(_):
    global queue
    if state['idx'] >= len(queue):
        return
    row = queue.iloc[state['idx']]
    targets = [row]
    for cb, sib in sibling_state['checkboxes']:
        if cb.value:
            targets.append(sib)
    for r in targets:
        add_review(r['Ticker'], r['Period_str'], r['Rule'], status_w.value, note_w.value)
        done.add(_key(r))
    queue = _rebuild_queue()
    render()


def on_skip(_):
    state['idx'] += 1
    render()


def on_apply(_):
    global queue
    state['sort'] = sort_w.value
    state['min_rel'] = float(min_rel_w.value)
    state['rules'] = list(rules_w.value) or None
    queue = _rebuild_queue()
    state['idx'] = 0
    render()


queue = _rebuild_queue()
container = widgets.VBox([])
queue_list_html = widgets.HTML()
note_w = widgets.Textarea(description='Note:', layout=widgets.Layout(width='800px', height='80px'))
status_w = widgets.Dropdown(options=['ok', 'data_error', 'to_check'], description='Status:', value='ok')
next_btn = widgets.Button(description='Mark Reviewed & Next', button_style='primary')
skip_btn = widgets.Button(description='Skip')
sibling_state = {'checkboxes': []}

next_btn.on_click(on_next)
skip_btn.on_click(on_skip)
apply_btn.on_click(on_apply)
render()

In [10]:
_flag_ticker_w = widgets.Text(description='Ticker:', value='')
_flag_period_w = widgets.Text(description='Period:', placeholder='2024FY or 2024Q2')
_flag_subject_w = widgets.Text(
    description='Subject:',
    placeholder='Revenue, Cash Flow Statement, Equity bridge, …',
    layout=widgets.Layout(width='600px'),
)
_flag_status_w = widgets.Dropdown(options=['ok', 'data_error', 'to_check'], description='Status:', value='to_check')
_flag_note_w = widgets.Textarea(description='Note:', layout=widgets.Layout(width='800px', height='60px'))
_flag_btn = widgets.Button(description='Add Flag', button_style='warning')
_flag_out = widgets.Output()


def _prefill_from_current():
    if state['idx'] < len(queue):
        row = queue.iloc[state['idx']]
        _flag_ticker_w.value = str(row['Ticker'])
        _flag_period_w.value = str(row['Period_str'])


def _on_add_flag(_):
    t = _flag_ticker_w.value.strip()
    p = _flag_period_w.value.strip()
    s = _flag_subject_w.value.strip()
    n = _flag_note_w.value.strip()
    with _flag_out:
        clear_output()
        if not (t and p and s):
            print('Need ticker, period, and subject.')
            return
        add_flag(t, p, s, _flag_status_w.value, n)
        print(f'Flagged: {t} {p} subject={s!r} status={_flag_status_w.value}')
        _flag_subject_w.value = ''
        _flag_note_w.value = ''


_flag_btn.on_click(_on_add_flag)
_prefill_from_current()

In [11]:
_view_ticker_w = widgets.Text(description='Ticker:', value='')
_view_stmt_w = widgets.Dropdown(options=['income', 'balance', 'cashflow'], description='Statement:', value='income')
_view_periods_w = widgets.Text(
    description='Periods:',
    placeholder='2024FY, 2023FY, 2024Q2  (blank = all)',
    layout=widgets.Layout(width='600px'),
)
_view_items_w = widgets.Text(
    description='Items:',
    placeholder='Revenue, Gross Profit  (blank = all)',
    layout=widgets.Layout(width='600px'),
)
_view_btn = widgets.Button(description='Show', button_style='info')
_view_out = widgets.Output()


def _prefill_viewer():
    if state['idx'] < len(queue):
        row = queue.iloc[state['idx']]
        _view_ticker_w.value = str(row['Ticker'])
        _view_periods_w.value = str(row['Period_str'])


def _on_view(_):
    t = _view_ticker_w.value.strip()
    periods_s = _view_periods_w.value.strip()
    items_s = _view_items_w.value.strip()
    periods = [p.strip() for p in periods_s.split(',') if p.strip()] or None
    items = [p.strip() for p in items_s.split(',') if p.strip()] or None
    with _view_out:
        clear_output()
        if not t:
            print('Need ticker.')
            return
        df = statement(t, _view_stmt_w.value, periods=periods, items=items)
        if df.empty:
            print('No data for that ticker/periods.')
            return
        display(df.style.format(_fmt_num))


_view_btn.on_click(_on_view)
_prefill_viewer()

## Load Queue

In [12]:
display(widgets.HBox([sort_w, min_rel_w]))
display(rules_w)
display(apply_btn)

SelectMultiple(description='Rules:', layout=Layout(width='400px'), options=('cash_chain', 'ni_income_vs_cashfl…

Button(button_style='info', description='Apply', style=ButtonStyle())

In [13]:
display(queue_list_html)

HTML(value='<p><b>28 unreviewed</b></p><table border="1" class="dataframe">\n  <thead>\n    <tr style="text-al…

## Reviewer

Single-finding loop. `Mark Reviewed & Next` appends to the TOML and advances. `Skip` advances without saving. Re-run `Load Queue` cell to refresh.

In [14]:
display(widgets.VBox([
    container,
    note_w,
    status_w,
    widgets.HBox([next_btn, skip_btn]),
]))

## Side-Findings

Flag a SimFin issue **not** caught by a rule. `Subject` is free-form scope — a single field, a full statement, or anything in between (e.g. `Revenue`, `Cash Flow Statement`, `Equity bridge`). Ticker + period prefill from the current review item; override if needed.

In [15]:
display(widgets.VBox([
    widgets.HBox([_flag_ticker_w, _flag_period_w]),
    _flag_subject_w,
    _flag_note_w,
    _flag_status_w,
    _flag_btn,
    _flag_out,
]))

## Statement Viewer

Pull income / balance / cashflow for a ticker across one or more periods. Periods comma-separated, e.g. `2024FY, 2023FY, 2024Q2`. Leave blank for all available.

In [16]:
display(widgets.VBox([
    widgets.HBox([_view_ticker_w, _view_stmt_w]),
    _view_periods_w,
    _view_items_w,
    _view_btn,
    _view_out,
]))

## Audit — All Reviews

In [17]:
_reviews = load_reviews_df()
print(f'{len(_reviews)} total entries (rule-reviews + manual flags)')
display(_reviews)

print()
_flags = load_flags_df()
print(f'{len(_flags)} manual flags')
display(_flags)

10 total entries (rule-reviews + manual flags)


,ticker,period,rule,status,note,reviewed_at,subject
0,AMZN,2024FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
1,AMZN,2023FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
2,AMZN,2022FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
3,AMZN,2021FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
4,AMZN,2020FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
5,JNJ,2024FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
6,JNJ,2023FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
7,JNJ,2021FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
8,JNJ,2020FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
9,META,2022FY,manual,to_check,,2026-05-17,Cashflow



1 manual flags


,ticker,period,rule,status,note,reviewed_at,subject
0,META,2022FY,manual,to_check,,2026-05-17,Cashflow


In [18]:
_reviews['note'].values

<ArrowStringArray>
['Missing Foreign currency effect on cash, cash equivalents, and restricted cash -1,301,000,000',    'Missing Foreign currency effect on cash, cash equivalents, and restricted cash 403,000,000',
 'Missing Foreign currency effect on cash, cash equivalents, and restricted cash -1,093,000,000',   'Missing Foreign currency effect on cash, cash equivalents, and restricted cash -364,000,000',
   'Missing Foreign currency effect on cash, cash equivalents, and restricted cash 618,000,000,',             'Missing Effect of exchange rate changes on cash and cash equivalents -289,000,000',
             'Missing Effect of exchange rate changes on cash and cash equivalents -112,000,000',             'Missing Effect of exchange rate changes on cash and cash equivalents -178,000,000',
              'Missing Effect of exchange rate changes on cash and cash equivalents -89,000,000',                                                                                              '']
Length